# HAM10000: two more seeds per split
Copyright (c) 2026 Janat Yasser. All rights reserved.

The first notebook trained one model per split (seed 42). This one adds seeds 1 and 2 for both splits, so the comparison rests on 3 runs each. Then it combines everything.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. It takes about 2 hours. It reads and adds to `MyDrive/ham10000_results`, and saves after every run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, filecmp
OUT = '/content/drive/MyDrive/ham10000_results'
assert os.path.exists(f'{OUT}/results/lesion_metrics.json'), 'Run the first notebook (seed 42) before this one.'
os.makedirs('/content/ham10000', exist_ok=True)
%cd /content/ham10000
!nvidia-smi --query-gpu=name,memory.total --format=csv
shutil.copytree(f'{OUT}/results', 'results', dirs_exist_ok=True)   # the seed-42 results
def save():
    for d in ['results', 'runs']:
        if os.path.exists(d):
            shutil.copytree(d, f'{OUT}/{d}', dirs_exist_ok=True)
    print('saved to Drive:', OUT)

## The code

In [ ]:
%%writefile common.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""Shared paths, class names and small helpers for every step."""
import json
import random
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
RAW = DATA / "raw"                       # downloaded zip/csv files
TRAIN_IMAGES = DATA / "train"            # ISIC2018_Task3_Training_Input
TEST_IMAGES = DATA / "test"              # ISIC2018_Task3_Test_Input
SPLITS = ROOT / "splits"
RUNS = ROOT / "runs"
RESULTS = ROOT / "results"

# Column order of the ISIC ground-truth CSVs. Index = class id.
CLASSES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
CLASS_NAMES = {
    "MEL": "Melanoma",
    "NV": "Melanocytic nevus",
    "BCC": "Basal cell carcinoma",
    "AKIEC": "Actinic keratosis / Bowen's",
    "BKL": "Benign keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular lesion",
}
MEL = CLASSES.index("MEL")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def read_ground_truth(csv_path: Path):
    """ISIC ground truth is one-hot (image, MEL, NV, ...). Returns a DataFrame with image + label (int)."""
    import pandas as pd
    df = pd.read_csv(csv_path)
    df["label"] = df[CLASSES].values.argmax(axis=1)
    return df[["image", "label"]]


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")


In [ ]:
%%writefile step1_download.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""
Step 1a: download the ISIC 2018 Task 3 data (HAM10000) from the official ISIC
bucket listed at https://challenge.isic-archive.com/data, then unzip it.

Safe to re-run: finished downloads are skipped, half-finished ones restart.
License: CC BY-NC 4.0 (non-commercial use, with attribution).
"""
import shutil
import sys
import time
import urllib.request
import zipfile

from common import RAW, TEST_IMAGES, TRAIN_IMAGES

BASE = "https://isic-archive.s3.amazonaws.com/challenges/2018/"
FILES = [
    "ISIC2018_Task3_Training_GroundTruth.zip",
    "ISIC2018_Task3_Training_LesionGroupings.csv",
    "ISIC2018_Task3_Test_GroundTruth.zip",
    "ISIC2018_Task3_Test_Input.zip",        # ~400 MB, 1,512 images
    "ISIC2018_Task3_Training_Input.zip",    # ~2.6 GB, 10,015 images
]


def download(name: str) -> None:
    target = RAW / name
    url = BASE + name
    with urllib.request.urlopen(urllib.request.Request(url, method="HEAD")) as r:
        size = int(r.headers["Content-Length"])
    if target.exists() and target.stat().st_size == size:
        print(f"have      {name} ({size / 2**20:,.0f} MB)", flush=True)
        return
    part = target.with_suffix(target.suffix + ".part")
    print(f"download  {name} ({size / 2**20:,.0f} MB)", flush=True)
    start, done, last = time.time(), 0, 0.0
    with urllib.request.urlopen(url) as r, open(part, "wb") as f:
        while chunk := r.read(1 << 20):
            f.write(chunk)
            done += len(chunk)
            if time.time() - last > 10:
                last = time.time()
                rate = done / max(1e-6, last - start) / 2**20
                print(f"          {done / size:6.1%}  {rate:5.1f} MB/s", flush=True)
    if part.stat().st_size != size:
        sys.exit(f"{name}: got {part.stat().st_size} bytes, expected {size}. Run again.")
    part.replace(target)


def unzip(name: str, dest, expected_jpgs: int | None) -> None:
    if expected_jpgs and dest.exists() and len(list(dest.glob("*.jpg"))) >= expected_jpgs:
        print(f"unzipped  {name}", flush=True)
        return
    print(f"unzip     {name}", flush=True)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(RAW / name) as z:
        for member in z.infolist():
            if member.is_dir() or member.filename.startswith("__MACOSX"):
                continue
            filename = member.filename.rsplit("/", 1)[-1]
            if not filename or filename.startswith("."):
                continue
            with z.open(member) as src, open(dest / filename, "wb") as out:
                shutil.copyfileobj(src, out)


if __name__ == "__main__":
    RAW.mkdir(parents=True, exist_ok=True)
    for name in FILES:
        download(name)
    unzip("ISIC2018_Task3_Training_GroundTruth.zip", RAW, None)
    unzip("ISIC2018_Task3_Test_GroundTruth.zip", RAW, None)
    unzip("ISIC2018_Task3_Test_Input.zip", TEST_IMAGES, 1512)
    unzip("ISIC2018_Task3_Training_Input.zip", TRAIN_IMAGES, 10015)
    print(f"train images: {len(list(TRAIN_IMAGES.glob('*.jpg')))}, test images: {len(list(TEST_IMAGES.glob('*.jpg')))}")
    print("DOWNLOAD FINISHED", flush=True)


In [ ]:
%%writefile step2_splits.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""
Step 2: two train/validation splits of the 10,015 training images.

  splits/lesion.csv  The honest one. Whole lesions go to one side, so no
                     validation image has a sibling photo in training.
                     Stratified by class so rare classes appear in both.
  splits/image.csv   The leaky one most tutorials use: a random split by
                     image. Kept only to measure how much it flatters the
                     validation score.

Both put ~15% of images in validation. The official test set is never
touched here.
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

from common import RAW, SPLITS, save_json, read_ground_truth

SEED = 42

if __name__ == "__main__":
    SPLITS.mkdir(exist_ok=True)
    df = read_ground_truth(RAW / "ISIC2018_Task3_Training_GroundTruth.csv")
    groups = pd.read_csv(RAW / "ISIC2018_Task3_Training_LesionGroupings.csv")
    df = df.merge(groups[["image", "lesion_id"]], on="image", validate="one_to_one")

    # Lesion-level: 1 fold of a 7-fold stratified group split is ~14-15% of images
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
    _, val_idx = next(sgkf.split(df, df["label"], groups=df["lesion_id"]))
    lesion = df.copy()
    lesion["split"] = "train"
    lesion.loc[lesion.index[val_idx], "split"] = "val"

    # Image-level (leaky): same validation size, stratified by class, ignoring lesions
    tr, va = train_test_split(df.index, test_size=len(val_idx), stratify=df["label"], random_state=SEED)
    image = df.copy()
    image["split"] = "train"
    image.loc[va, "split"] = "val"

    report = {}
    for name, s in [("lesion", lesion), ("image", image)]:
        s.to_csv(SPLITS / f"{name}.csv", index=False)
        train_lesions = set(s.loc[s.split == "train", "lesion_id"])
        val = s[s.split == "val"]
        leaked = val["lesion_id"].isin(train_lesions)
        report[name] = {
            "train_images": int((s.split == "train").sum()),
            "val_images": int(len(val)),
            "val_images_whose_lesion_is_also_in_train": int(leaked.sum()),
            "share_leaked": round(float(leaked.mean()), 4),
            "val_class_counts": val["label"].value_counts().sort_index().tolist(),
        }
    assert report["lesion"]["val_images_whose_lesion_is_also_in_train"] == 0
    save_json(report, SPLITS / "splits_report.json")
    for k, v in report.items():
        print(f"{k:6s} train {v['train_images']:5d}  val {v['val_images']:5d}  "
              f"val images with a sibling in train: {v['val_images_whose_lesion_is_also_in_train']} ({v['share_leaked']:.1%})")
    print("SPLITS FINISHED", flush=True)


In [ ]:
%%writefile step3_train.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""
Step 3: fine-tune an ImageNet-pretrained EfficientNet-B0 on one split.

    python step3_train.py --split lesion      # the honest split
    python step3_train.py --split image       # the leaky split, for comparison

Choices, and why:
  - EfficientNet-B0 (5.3M parameters) at 224x224 with mixed precision fits
    comfortably in 4 GB of GPU memory at batch size 32.
  - Class-weighted cross-entropy: nevi are ~67% of the data, so without
    weights the model can score well by mostly saying "nevus".
    Weights are 1/sqrt(class frequency), a middle ground between none and
    full inverse frequency (which over-corrects for the 115 dermatofibromas).
  - The best epoch is picked by validation *balanced accuracy* (mean recall
    over the 7 classes), not plain accuracy, for the same reason.
  - Augmentation: flips in both directions and 90-degree-ish rotations
    (a lesion has no "up"), mild crops and colour jitter.
Outputs go to runs/<split>/ (or runs/<run-name>/): best.pt, history.csv, config.json.
"""
import argparse
import csv
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from common import CLASSES, RUNS, SPLITS, TRAIN_IMAGES, seed_everything

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def make_transforms(size: int):
    train = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(90),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    evaluate = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train, evaluate


class LesionImages(Dataset):
    def __init__(self, folder, names, labels, transform):
        self.folder, self.names, self.labels, self.transform = folder, list(names), list(labels), transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        img = Image.open(self.folder / f"{self.names[i]}.jpg").convert("RGB")
        return self.transform(img), self.labels[i]


def build_model(pretrained: bool) -> nn.Module:
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return m


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    out, ys = [], []
    for x, y in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            out.append(model(x.to(device, non_blocking=True).to(memory_format=torch.channels_last)).float().cpu())
        ys.append(y)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()


def balanced_accuracy(logits, y):
    pred = logits.argmax(1)
    recalls = [(pred[y == c] == c).mean() for c in range(len(CLASSES)) if (y == c).any()]
    return float(np.mean(recalls))


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--split", choices=["lesion", "image"], required=True)
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--lr", type=float, default=4e-4)
    p.add_argument("--size", type=int, default=224)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no-pretrained", action="store_true", help="for quick offline tests only")
    p.add_argument("--limit", type=int, default=0, help="use only N training images (tests only)")
    p.add_argument("--run-name", default="", help="output folder under runs/ (default: the split name)")
    args = p.parse_args()

    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = RUNS / (args.run_name or args.split)
    out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(SPLITS / f"{args.split}.csv")
    train_df, val_df = df[df.split == "train"], df[df.split == "val"]
    if args.limit:
        train_df = train_df.sample(min(args.limit, len(train_df)), random_state=args.seed)
        val_df = val_df.sample(min(max(args.limit // 4, 8), len(val_df)), random_state=args.seed)

    t_train, t_eval = make_transforms(args.size)
    loader_kw = dict(num_workers=args.workers, pin_memory=device.type == "cuda", persistent_workers=args.workers > 0)
    train_loader = DataLoader(LesionImages(TRAIN_IMAGES, train_df.image, train_df.label, t_train),
                              batch_size=args.batch_size, shuffle=True, drop_last=True, **loader_kw)
    val_loader = DataLoader(LesionImages(TRAIN_IMAGES, val_df.image, val_df.label, t_eval),
                            batch_size=args.batch_size * 2, shuffle=False, **loader_kw)

    counts = np.bincount(train_df.label, minlength=len(CLASSES)).astype(float)
    weights = 1 / np.sqrt(np.maximum(counts, 1))
    weights = weights / (weights * counts).sum() * counts.sum()   # keeps the average loss scale
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))

    model = build_model(not args.no_pretrained).to(device).to(memory_format=torch.channels_last)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    steps = args.epochs * len(train_loader)
    warmup = len(train_loader)   # one epoch
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, steps - warmup))))
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

    config = {**vars(args), "device": str(device), "train_images": len(train_df), "val_images": len(val_df),
              "class_weights": [round(w, 4) for w in weights], "torch": torch.__version__,
              "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None}
    (out / "config.json").write_text(json.dumps(config, indent=2))
    print(json.dumps(config, indent=2), flush=True)

    best, history = -1.0, []
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item() * len(y)
            n += len(y)
        logits, ys = predict(model, val_loader, device)
        val_loss = float(nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(ys)).item())
        bacc = balanced_accuracy(logits, ys)
        acc = float((logits.argmax(1) == ys).mean())
        row = {"epoch": epoch, "train_loss": round(total / n, 4), "val_loss": round(val_loss, 4),
               "val_acc": round(acc, 4), "val_balanced_acc": round(bacc, 4), "seconds": round(time.time() - t0, 1)}
        if device.type == "cuda":
            row["gpu_mem_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        history.append(row)
        if bacc > best:
            best = bacc
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_balanced_acc": bacc, "config": config}, out / "best.pt")
            row["saved"] = True
        print(row, flush=True)

    with open(out / "history.csv", "w", newline="") as f:
        fields = list(dict.fromkeys(k for r in history for k in r))   # keys in order of first appearance
        w = csv.DictWriter(f, fieldnames=fields, restval="")
        w.writeheader()
        w.writerows(history)
    print(f"TRAINING FINISHED: best validation balanced accuracy {best:.4f}", flush=True)


if __name__ == "__main__":   # needed on Windows, where DataLoader workers re-import this file
    main()


In [ ]:
%%writefile step4_evaluate.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""
Step 4: evaluate a trained model on the official ISIC 2018 test set
(1,512 images it has never seen, from a different collection period).

    python step4_evaluate.py --split lesion
    python step4_evaluate.py --split image
    python step4_evaluate.py --compare        # after both: results/summary.md

Rules that keep the test set honest:
  - Anything that is *chosen* (the temperature for calibration, the melanoma
    decision threshold) is chosen on the validation split, then applied to
    the test set unchanged.
  - Every headline number gets a 95% bootstrap interval (2,000 resamples of
    the test images), because 1,512 images, and only ~170 melanomas, leave
    real uncertainty.
"""
import argparse
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, roc_auc_score
from torch.utils.data import DataLoader

from common import CLASSES, MEL, RAW, RESULTS, RUNS, SPLITS, TEST_IMAGES, TRAIN_IMAGES, read_ground_truth, save_json
from step3_train import LesionImages, build_model, make_transforms, predict

TARGET_SPECIFICITY = 0.90   # melanoma operating point, chosen on validation
BOOTSTRAP = 2000


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y):
    """One scalar T minimising validation NLL (Guo et al., 2017)."""
    t = torch.nn.Parameter(torch.ones(1))
    lg, yt = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(lg / t.clamp(min=0.05), yt)
        loss.backward()
        return loss
    opt.step(closure)
    return float(t.clamp(min=0.05).item())


def ece(probs, y, bins=15):
    """Expected calibration error of the top-class confidence."""
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(total)


def metrics(probs, y, mel_threshold):
    pred = probs.argmax(1)
    recall = {c: float((pred[y == i] == i).mean()) for i, c in enumerate(CLASSES) if (y == i).any()}
    is_mel = y == MEL
    flag = probs[:, MEL] >= mel_threshold
    return {
        "accuracy": float((pred == y).mean()),
        "balanced_accuracy": float(np.mean(list(recall.values()))),
        "recall_per_class": recall,
        "macro_auc": float(roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=list(range(len(CLASSES))))),
        "melanoma_auc": float(roc_auc_score(is_mel, probs[:, MEL])),
        "melanoma_sensitivity": float(flag[is_mel].mean()),
        "melanoma_specificity": float((~flag[~is_mel]).mean()),
    }


def bootstrap(probs, y, mel_threshold, keys, seed=0):
    rng = np.random.default_rng(seed)
    draws = {k: [] for k in keys}
    for _ in range(BOOTSTRAP):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < len(CLASSES):
            continue
        m = metrics(probs[i], y[i], mel_threshold)
        for k in keys:
            draws[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in draws.items()}


def reliability_plot(probs_before, probs_after, y, path, title):
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    for probs, label, color in [(probs_before, "as trained", "#ee6c4d"), (probs_after, "after temperature scaling", "#3d5a80")]:
        conf, correct = probs.max(1), probs.argmax(1) == y
        edges = np.linspace(0, 1, 11)
        xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum() >= 10:
                xs.append(conf[m].mean())
                ys.append(correct[m].mean())
        ax.plot(xs, ys, "o-", label=f"{label} (ECE {ece(probs, y):.3f})", color=color)
    ax.plot([0, 1], [0, 1], "--", color="#999", lw=1)
    ax.set_xlabel("confidence")
    ax.set_ylabel("share correct")
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=150)


def confusion_plot(probs, y, path, title):
    cm = confusion_matrix(y, probs.argmax(1), labels=list(range(len(CLASSES))), normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4.4))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 0.5 else "#222")
    ax.set_xticks(range(len(CLASSES)), CLASSES, fontsize=8)
    ax.set_yticks(range(len(CLASSES)), CLASSES, fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)


def evaluate(split, workers, run_name=""):
    run = run_name or split
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(RUNS / run / "best.pt", map_location=device, weights_only=False)
    size = ckpt["config"]["size"]
    model = build_model(False).to(device).to(memory_format=torch.channels_last)
    model.load_state_dict(ckpt["model"])
    _, t_eval = make_transforms(size)

    df = pd.read_csv(SPLITS / f"{split}.csv")
    val = df[df.split == "val"]
    if ckpt["config"].get("limit"):
        val = val.sample(min(max(ckpt["config"]["limit"] // 4, 8), len(val)), random_state=ckpt["config"]["seed"])
    test = read_ground_truth(RAW / "ISIC2018_Task3_Test_GroundTruth.csv")
    test = test[test.image.map(lambda n: (TEST_IMAGES / f"{n}.jpg").exists())]
    kw = dict(batch_size=64, num_workers=workers)
    val_logits, val_y = predict(model, DataLoader(LesionImages(TRAIN_IMAGES, val.image, val.label, t_eval), **kw), device)
    test_logits, test_y = predict(model, DataLoader(LesionImages(TEST_IMAGES, test.image, test.label, t_eval), **kw), device)

    # chosen on validation only
    T = fit_temperature(val_logits, val_y)
    val_p, test_p_raw, test_p = softmax(val_logits / T), softmax(test_logits), softmax(test_logits / T)
    neg = np.sort(val_p[val_y != MEL, MEL])
    mel_threshold = float(neg[int(np.ceil(TARGET_SPECIFICITY * len(neg))) - 1]) if len(neg) else 0.5

    val_m = metrics(val_p, val_y, mel_threshold)
    test_m = metrics(test_p, test_y, mel_threshold)
    ci = bootstrap(test_p, test_y, mel_threshold, ["balanced_accuracy", "macro_auc", "melanoma_auc", "melanoma_sensitivity", "melanoma_specificity"])

    RESULTS.mkdir(exist_ok=True)
    np.savez_compressed(RESULTS / f"{run}_predictions.npz", test_logits=test_logits, test_y=test_y,
                        test_images=np.array(test.image), val_logits=val_logits, val_y=val_y, temperature=T)
    out = {
        "split": split, "run": run, "seed": ckpt["config"]["seed"], "best_epoch": ckpt["epoch"], "temperature": T,
        "melanoma_threshold": mel_threshold, "target_specificity_on_val": TARGET_SPECIFICITY,
        "validation": val_m, "test": test_m, "test_95ci": ci,
        "test_ece_before": ece(test_p_raw, test_y), "test_ece_after": ece(test_p, test_y),
        "n_val": int(len(val_y)), "n_test": int(len(test_y)), "n_test_melanoma": int((test_y == MEL).sum()),
    }
    save_json(out, RESULTS / f"{run}_metrics.json")
    confusion_plot(test_p, test_y, RESULTS / f"{run}_confusion.png", f"Official test set, {split}-level split")
    reliability_plot(test_p_raw, test_p, test_y, RESULTS / f"{run}_reliability.png", f"Calibration on the test set ({split} split)")
    print(json.dumps({k: out[k] for k in ["split", "best_epoch", "temperature", "test_ece_before", "test_ece_after"]}, indent=2))
    for name, m in [("validation", val_m), ("test", test_m)]:
        print(f"{name:10s} balanced acc {m['balanced_accuracy']:.3f}  macro AUC {m['macro_auc']:.3f}  "
              f"melanoma AUC {m['melanoma_auc']:.3f}  sens {m['melanoma_sensitivity']:.3f} at spec {m['melanoma_specificity']:.3f}")
    print(f"EVALUATION FINISHED ({run})", flush=True)


def compare():
    r = {s: json.loads((RESULTS / f"{s}_metrics.json").read_text()) for s in ["lesion", "image"]}
    def f(x): return f"{x:.3f}"
    def ci(s, k): lo, hi = r[s]["test_95ci"][k]; return f"{r[s]['test'][k]:.3f} ({lo:.3f}–{hi:.3f})"
    lines = [
        "# HAM10000: does the split matter?", "",
        "Same model, same training recipe; only the train/validation split differs. "
        "The official ISIC 2018 test set (1,512 images) is untouched by both.", "",
        "| | Split by lesion (honest) | Split by image (leaky) |", "|---|---|---|",
        f"| Validation balanced accuracy | {f(r['lesion']['validation']['balanced_accuracy'])} | {f(r['image']['validation']['balanced_accuracy'])} |",
        f"| **Test** balanced accuracy (95% CI) | {ci('lesion', 'balanced_accuracy')} | {ci('image', 'balanced_accuracy')} |",
        f"| Validation − test gap | {f(r['lesion']['validation']['balanced_accuracy'] - r['lesion']['test']['balanced_accuracy'])} | {f(r['image']['validation']['balanced_accuracy'] - r['image']['test']['balanced_accuracy'])} |",
        f"| Test macro AUC | {ci('lesion', 'macro_auc')} | {ci('image', 'macro_auc')} |",
        f"| Test melanoma AUC | {ci('lesion', 'melanoma_auc')} | {ci('image', 'melanoma_auc')} |",
        f"| Test melanoma sensitivity at the val-chosen 90%-specificity threshold | {ci('lesion', 'melanoma_sensitivity')} | {ci('image', 'melanoma_sensitivity')} |",
        f"| …achieved test specificity | {ci('lesion', 'melanoma_specificity')} | {ci('image', 'melanoma_specificity')} |",
        f"| Test ECE before → after temperature scaling | {f(r['lesion']['test_ece_before'])} → {f(r['lesion']['test_ece_after'])} | {f(r['image']['test_ece_before'])} → {f(r['image']['test_ece_after'])} |",
        "",
        "Read the gap row: a validation score far above the test score means the validation set was too easy, here because of sibling photos.",
    ]
    (RESULTS / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines))
    print("COMPARE FINISHED", flush=True)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--split", choices=["lesion", "image"])
    p.add_argument("--compare", action="store_true")
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--run-name", default="", help="which runs/<run-name>/ to evaluate (default: the split name)")
    a = p.parse_args()
    compare() if a.compare else evaluate(a.split, a.workers, a.run_name)


In [ ]:
%%writefile step5_seeds.py
# Copyright (c) 2026 Janat Yasser. All rights reserved.
"""
Step 5: combine the runs of both splits over several seeds.

One seed can't separate "this split is better" from "this run got lucky".
So each split is trained with 3 seeds (42, 1, 2), and this step reports:
  - mean ± std over seeds of every headline number, for each split,
  - the validation-minus-test gap per seed (the leak's signature),
  - a paired bootstrap on the official test images comparing the two
    splits' seed-averaged predictions (each run's probabilities after its own
    temperature scaling, averaged over seeds).

    python step5_seeds.py      ->  results/seeds_summary.md, results/seeds_gap.png
"""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from common import CLASSES, RESULTS, save_json
from step4_evaluate import softmax

SPLITS_ = ["lesion", "image"]
LABEL = {"lesion": "Split by lesion (honest)", "image": "Split by image (leaky)"}
COLOR = {"lesion": "#3d5a80", "image": "#ee6c4d"}


def runs(split):
    out = []
    for f in sorted(RESULTS.glob(f"{split}*_metrics.json")):
        m = json.loads(f.read_text())
        if m["split"] != split:
            continue
        p = np.load(f.with_name(f.name.replace("_metrics.json", "_predictions.npz")), allow_pickle=True)   # our own files; image names are stored as objects
        out.append((m, softmax(p["test_logits"] / float(p["temperature"])), p["test_y"], list(p["test_images"])))
    return out


def balanced_acc(y, pred):
    return float(np.mean([(pred[y == c] == c).mean() for c in range(len(CLASSES)) if (y == c).any()]))


def main():
    R = {s: runs(s) for s in SPLITS_}
    for s in SPLITS_:
        assert R[s], f"no runs found for {s}"
    images = R["lesion"][0][3]
    for s in SPLITS_:
        for _, _, _, im in R[s]:
            assert im == images, "test image order differs between runs"
    y = R["lesion"][0][2]

    def stat(s, f):
        v = np.array([f(m) for m, *_ in R[s]])
        return v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0, v

    rows = [
        ("Validation balanced accuracy", lambda m: m["validation"]["balanced_accuracy"]),
        ("**Test** balanced accuracy", lambda m: m["test"]["balanced_accuracy"]),
        ("Validation − test gap", lambda m: m["validation"]["balanced_accuracy"] - m["test"]["balanced_accuracy"]),
        ("Test macro AUC", lambda m: m["test"]["macro_auc"]),
        ("Test melanoma AUC", lambda m: m["test"]["melanoma_auc"]),
        ("Test melanoma sensitivity (val-chosen 90%-spec threshold)", lambda m: m["test"]["melanoma_sensitivity"]),
        ("…achieved test specificity", lambda m: m["test"]["melanoma_specificity"]),
        ("Test ECE after temperature scaling", lambda m: m["test_ece_after"]),
    ]
    seeds = {s: [m.get("seed", 42) for m, *_ in R[s]] for s in SPLITS_}   # the first runs (seed 42) predate the "seed" field
    lines = ["# HAM10000: does the split matter? (several seeds)", "",
             f"Seeds: lesion {seeds['lesion']}, image {seeds['image']}. Mean ± std over seeds; "
             "the official test set (1,512 images) is untouched by every run.", "",
             f"| | {LABEL['lesion']} | {LABEL['image']} |", "|---|---|---|"]
    table = {}
    for name, f in rows:
        cells = []
        for s in SPLITS_:
            mu, sd, v = stat(s, f)
            table.setdefault(name, {})[s] = {"mean": mu, "std": sd, "per_seed": v.tolist()}
            cells.append(f"{mu:.3f} ± {sd:.3f}")
        lines.append(f"| {name} | {cells[0]} | {cells[1]} |")

    # Seed-averaged ("ensemble") predictions, compared on the same test images
    ens = {s: np.mean([p for _, p, _, _ in R[s]], axis=0).argmax(1) for s in SPLITS_}
    rng = np.random.default_rng(0)
    d = []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        d.append(balanced_acc(y[i], ens["lesion"][i]) - balanced_acc(y[i], ens["image"][i]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    ens_bacc = {s: balanced_acc(y, ens[s]) for s in SPLITS_}
    gap = {s: table["Validation − test gap"][s]["per_seed"] for s in SPLITS_}
    lines += ["",
              f"- **Seed-averaged models on the test set:** lesion {ens_bacc['lesion']:.3f}, image {ens_bacc['image']:.3f}; "
              f"difference (lesion − image) 95% CI [{lo:+.3f}, {hi:+.3f}]"
              + (" → the honest split is better" if lo > 0 else " → the leaky split is better" if hi < 0 else " → no clear difference on test"),
              f"- **Gap per seed:** lesion {', '.join(f'{g:.3f}' for g in gap['lesion'])}; "
              f"image {', '.join(f'{g:.3f}' for g in gap['image'])}. "
              + ("Every leaky run overstates its test score by more than every honest run."
                 if min(gap["image"]) > max(gap["lesion"]) else "The gaps overlap between splits."),
              ""]
    (RESULTS / "seeds_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    save_json({"table": table, "seed_averaged_test_bacc": ens_bacc, "diff_ci": [float(lo), float(hi)]}, RESULTS / "seeds_summary.json")

    # Figure: validation -> test for every run (the slope is the leak)
    fig, ax = plt.subplots(figsize=(4.6, 4))
    for s in SPLITS_:
        for k, (m, *_rest) in enumerate(R[s]):
            v, t = m["validation"]["balanced_accuracy"], m["test"]["balanced_accuracy"]
            ax.plot([0, 1], [v, t], "o-", color=COLOR[s], alpha=0.85, label=LABEL[s] if k == 0 else None)
    ax.set_xticks([0, 1], ["validation", "official test"])
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylabel("balanced accuracy")
    ax.set_title("Same model, two ways to split: score on validation vs test", fontsize=9)
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(RESULTS / "seeds_gap.png", dpi=150)
    print("\n".join(lines))
    print("SEEDS FINISHED", flush=True)


if __name__ == "__main__":
    main()


## Data (download, shrink to 256 px as before), and the same splits
The splits are fixed by `random_state=42`, so they come out identical. The cell checks that against the saved copy.

In [ ]:
!python step1_download.py
from PIL import Image as PILImage
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
def shrink(p):
    im = PILImage.open(p)
    if min(im.size) > 256:
        im.thumbnail((10_000, 256) if im.width > im.height else (256, 10_000))
        im.convert('RGB').save(p, quality=95)
for d in ['data/train', 'data/test']:
    files = list(Path(d).glob('*.jpg'))
    with ThreadPoolExecutor(8) as ex: list(ex.map(shrink, files))
    print(d, len(files), 'images resized')
!python step2_splits.py
for s in ['lesion', 'image']:
    assert filecmp.cmp(f'splits/{s}.csv', f'{OUT}/splits/{s}.csv', shallow=False), f'{s} split differs from the first run'
print('splits identical to the first run')

## Train and evaluate: seeds 1 and 2 for both splits (about 28 min per run)

In [ ]:
for seed in [1, 2]:
    for split in ['lesion', 'image']:
        run = f'{split}_s{seed}'
        if os.path.exists(f'{OUT}/results/{run}_metrics.json'):
            print('already done:', run); continue
        !python step3_train.py --split {split} --seed {seed} --run-name {run} --workers 2
        !python step4_evaluate.py --split {split} --run-name {run} --workers 2
        save()

## Combined results

In [ ]:
!python step5_seeds.py
save()
from IPython.display import Image, Markdown, display
display(Markdown(open('results/seeds_summary.md', encoding='utf-8').read()))
display(Image('results/seeds_gap.png', width=420))